In [39]:
import sqlite3
import pandas as pd
from transformers import pipeline

Реальный код

In [40]:
conn = sqlite3.connect("shop.db")
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    age INTEGER,
    registration_date DATE
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product TEXT,
    quantity INTEGER,
    price REAL,
    order_date DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")


In [41]:
customers_data = [
    (1, 'Иван Петров', 'Москва', 28, '2023-01-15'),
    (2, 'Мария Сидорова', 'Москва', 32, '2023-02-20'),
    (3, 'Алексей Смирнов', 'Санкт-Петербург', 45, '2023-03-10'),
    (4, 'Елена Козлова', 'Казань', 25, '2023-04-05'),
    (5, 'Дмитрий Иванов', 'Москва', 38, '2023-05-12')
]

orders_data = [
    (1, 1, 'iPhone 15', 1, 79900, '2024-01-10'),
    (2, 1, 'Чехол', 2, 1500, '2024-01-10'),
    (3, 2, 'Samsung Galaxy', 1, 64900, '2024-01-15'),
    (4, 2, 'Наушники', 1, 5000, '2024-01-15'),
    (5, 3, 'MacBook Pro', 1, 149900, '2024-01-20'),
    (6, 4, 'Xiaomi Mi Band', 2, 3000, '2024-01-25'),
    (7, 1, 'Чехол', 1, 750, '2024-02-01'),
    (8, 5, 'iPad Air', 1, 59900, '2024-02-05')
]

In [42]:
cursor.executemany("INSERT OR REPLACE INTO customers VALUES (?,?,?,?,?)", customers_data)
cursor.executemany("INSERT OR REPLACE INTO orders VALUES (?,?,?,?,?,?)", orders_data)
conn.commit()

In [43]:
print(pd.read_sql("SELECT * FROM customers", conn))

   customer_id             name             city  age registration_date
0            1      Иван Петров           Москва   28        2023-01-15
1            2   Мария Сидорова           Москва   32        2023-02-20
2            3  Алексей Смирнов  Санкт-Петербург   45        2023-03-10
3            4    Елена Козлова           Казань   25        2023-04-05
4            5   Дмитрий Иванов           Москва   38        2023-05-12


In [44]:
print(pd.read_sql("SELECT * FROM orders", conn))

   order_id  customer_id         product  quantity     price  order_date
0         1            1       iPhone 15         1   79900.0  2024-01-10
1         2            1           Чехол         2    1500.0  2024-01-10
2         3            2  Samsung Galaxy         1   64900.0  2024-01-15
3         4            2        Наушники         1    5000.0  2024-01-15
4         5            3     MacBook Pro         1  149900.0  2024-01-20
5         6            4  Xiaomi Mi Band         2    3000.0  2024-01-25
6         7            1           Чехол         1     750.0  2024-02-01
7         8            5        iPad Air         1   59900.0  2024-02-05


Transformer

In [46]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [48]:
model_name = "mrm8488/t5-base-finetuned-wikiSQL"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

In [54]:
def text_to_sql(question, table_schema):
  input_text = f"translate English to SQL: {question} | {table_schema}"
  inputs = tokenizer(input_text,return_tensors="pt", max_length=512,truncation=True)
  outputs = model.generate(
      inputs["input_ids"],
      max_length=256,
      num_beams=4,
      early_stopping=True
  )
  sql = tokenizer.decode(outputs[0],skip_special_tokens=True)
  return sql

In [55]:
schema = """
Таблица customers: customer_id (int), name (text), city (text), age (int), registration_date (date)
Таблица orders: order_id (int), customer_id (int), product (text), quantity (int), price (real), order_date (date)
"""

In [56]:
questions = [
    "Show all customers from Moscow",
    "How many orders did Ivan Petrov make?",
    "Show total revenue by customer",
    "Find customers older than 30",
    "What products were ordered in February 2024?"
]


In [57]:
if __name__ == '__main__':
  for q in questions:
    sql = text_to_sql(q, schema)
    print(f"Вопрос: {q}")
    print(f"SQL: {sql}\n")

Вопрос: Show all customers from Moscow
SQL: SELECT Customers FROM table WHERE City = moscow | алиа customers: customer_id (int), name (text), city (text), age (int), registration_date (date) алиа orders: order_id (int), customer_id (int), product (text), quantity (int), price (real), order_date (date)

Вопрос: How many orders did Ivan Petrov make?
SQL: SELECT Orders FROM table WHERE Name = ivan petrov | алиа customers: customer_id (int), name (text), city (text), age (int), registration_date (date) алиа orders: order_id (int), customer_id (int), product (text), quantity (int), price (real), order_date (date)

Вопрос: Show total revenue by customer
SQL: SELECT COUNT Revenue by customer | алиа customers: customer_id (int), name (text), city (text), age (int), registration_date (date) алиа orders: order_id (int), customer_id (int), product (text), quantity (int), price (real), order_date (date)

Вопрос: Find customers older than 30
SQL: SELECT Customers > 30 | алиа customers: customer_id 